# Visualization Layer Preparation - Gold Layer

Prepares optimized viz tables for the React app with pre-computed metrics.

**Order of creation** (dependencies flow downward):
1. `viz_h3_grid` - H3 hexagon grid for Massachusetts
2. `viz_competitors` - Pizza competitor locations
3. `viz_existing_stores` - Current LCE stores with isochrones
4. `viz_partners` - Partner isochrones with candidate counts
5. `viz_expansion_candidates` - Candidates with distances, partner/competitor proximity
6. `viz_network_metrics` - Aggregate KPIs
7. `viz_optimization_results` - Pre-computed optimization (8 param combos)
8. **Genie Space** - Natural language analytics (optional)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, explode, lit, when
from pyspark.sql.window import Window
from datetime import datetime
from itertools import product
import numpy as np

# Parameters
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "geo_bronze")
dbutils.widgets.text("silver_schema", "geo_silver")
dbutils.widgets.text("gold_schema", "geo_gold")
dbutils.widgets.text("sql_warehouse_id", "")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")
sql_warehouse_id = dbutils.widgets.get("sql_warehouse_id")

assert catalog, "ERROR: catalog parameter is required"
print(f"Target: {catalog}.{gold_schema}")

## 1. H3 Grid (Massachusetts)

In [ ]:
ma_boundary = spark.table(f"{catalog}.{bronze_schema}.census_states").filter(
    (col("state_abbr") == "MA") | (col("state_fips") == "25")
)

viz_h3_grid = (
    ma_boundary
    .select(explode(expr("h3_coverash3string(ST_AsBinary(geometry), 5)")).alias("coarse_h3"))
    .select(explode(expr("h3_tochildren(coarse_h3, 8)")).alias("h3_cell_id"))
    .distinct()
    .withColumn("geometry", expr("ST_GeomFromWKT(h3_boundaryaswkt(h3_cell_id), 4326)"))
    .withColumn("center_lat", expr("ST_Y(ST_GeomFromWKT(h3_centeraswkt(h3_cell_id), 4326))"))
    .withColumn("center_lon", expr("ST_X(ST_GeomFromWKT(h3_centeraswkt(h3_cell_id), 4326))"))
)

viz_h3_grid.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_h3_grid")
print(f"✓ viz_h3_grid: {viz_h3_grid.count():,} cells")

## 2. Competitors

In [ ]:
viz_competitors = (
    spark.table(f"{catalog}.{silver_schema}.pois_competitors")
    .select(
        col("poi_id").alias("id"), "name", "latitude", "longitude",
        "poi_category", "poi_subcategory", "address"
    )
    .withColumn("marker_type", lit("competitor"))
    .withColumn("geometry", expr("ST_Point(longitude, latitude)"))
    .withColumn("h3_cell_id", expr("h3_longlatash3string(longitude, latitude, 8)"))
    .withColumn("geometry_geojson", expr("ST_AsGeoJSON(ST_Point(longitude, latitude))"))
)

viz_competitors.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_competitors")
print(f"✓ viz_competitors: {viz_competitors.count():,} rows")

## 3. Existing Stores

In [ ]:
# Load stores and join with isochrones
existing_stores = spark.table(f"{catalog}.{gold_schema}.current_stores_features_agg")
existing_cols = existing_stores.columns

# Filter to MA only
existing_stores_ma = existing_stores.filter(
    (col("state") == "MA") | (col("state") == "Massachusetts") | col("state").isNull()
)

# Join with isochrones
lce_isochrones = spark.table(f"{catalog}.{silver_schema}.isochrones_lce").select(
    col("location_id").alias("iso_store_number"),
    col("geometry").alias("isochrone_geometry")
)

existing_with_iso = existing_stores_ma.join(
    lce_isochrones,
    existing_stores_ma["store_number"] == lce_isochrones["iso_store_number"],
    "left"
).drop("iso_store_number")

# Calculate POI count
poi_cats = ['retail', 'food_drink', 'leisure', 'education', 'healthcare', 'financial', 'tourism', 'transportation']
poi_sum = sum([F.coalesce(col(c), lit(0)) for c in poi_cats if c in existing_cols], lit(0))

viz_existing = (
    existing_with_iso
    .select(
        "store_number", "latitude", "longitude",
        col("store_type") if "store_type" in existing_cols else lit("LCE").alias("store_type"),
        "city", lit("MA").alias("state"), "population", "annual_sales",
        poi_sum.alias("poi_count") if "total_poi_count" not in existing_cols else col("total_poi_count").alias("poi_count"),
        "geometry", "isochrone_geometry"
    )
    .withColumn("marker_type", lit("existing_lce"))
    .withColumn("h3_cell_id", expr("h3_longlatash3string(longitude, latitude, 8)"))
    .withColumn("geometry_geojson", expr("ST_AsGeoJSON(geometry)"))
    .withColumn("isochrone_geojson", when(col("isochrone_geometry").isNotNull(), expr("ST_AsGeoJSON(isochrone_geometry)")))
    .drop("isochrone_geometry")
    # Genie columns
    .withColumn("region", when((col("latitude") > 42.3) & (col("longitude") > -71.5), "Boston Metro")
        .when((col("latitude") > 42.0) & (col("longitude") > -72.0), "Greater Boston")
        .when(col("longitude") < -72.0, "Western MA").otherwise("Cape & Islands"))
    .withColumn("sales_rank", F.row_number().over(Window.orderBy(F.desc("annual_sales"))))
    .withColumn("sales_percentile", F.percent_rank().over(Window.orderBy("annual_sales")))
    .withColumn("performance_tier", when(col("sales_percentile") >= 0.75, "Top Performer")
        .when(col("sales_percentile") >= 0.50, "Above Average")
        .when(col("sales_percentile") >= 0.25, "Average").otherwise("Below Average"))
)

viz_existing.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_existing_stores")
print(f"✓ viz_existing_stores: {viz_existing.count():,} rows")

## 4. Partners

In [ ]:
partners = spark.table(f"{catalog}.{silver_schema}.isochrones_partners")
candidates_for_partners = spark.table(f"{catalog}.{gold_schema}.candidates_finalized").withColumn(
    "h3_cell_id", expr("h3_longlatash3string(longitude, latitude, 8)")
)

# Spatial join: candidates within partner isochrones
candidates_with_point = candidates_for_partners.withColumn(
    "candidate_point", expr("ST_SetSRID(ST_Point(longitude, latitude), 4326)")
).select("candidate_id", "h3_cell_id", "candidate_point", "predicted_annual_sales")

partners_with_candidates = partners.alias("p").join(
    candidates_with_point.alias("c"),
    expr("ST_Contains(p.geometry, c.candidate_point)"), "left"
)

candidate_agg = partners_with_candidates.groupBy(col("p.location_id")).agg(
    F.count("c.candidate_id").alias("candidate_count_in_isochrone"),
    F.sum("c.predicted_annual_sales").alias("total_candidate_sales_in_isochrone"),
    F.collect_list("c.h3_cell_id").alias("candidate_h3_cells_in_isochrone")
)

viz_partners = (
    partners.alias("p")
    .join(candidate_agg.alias("a"), col("p.location_id") == col("a.location_id"), "left")
    .select(
        col("p.location_id").alias("id"), col("p.store_type"), col("p.store_type").alias("name"),
        col("p.latitude"), col("p.longitude"), col("p.city"), col("p.state"),
        col("p.drive_time_minutes"), col("p.area_sqkm"), col("p.geometry"),
        F.coalesce(col("a.candidate_count_in_isochrone"), lit(0)).alias("candidate_count_in_isochrone"),
        F.coalesce(col("a.total_candidate_sales_in_isochrone"), lit(0)).alias("total_candidate_sales_in_isochrone"),
        F.coalesce(col("a.candidate_h3_cells_in_isochrone"), F.array()).alias("candidate_h3_cells_in_isochrone")
    )
    .withColumn("marker_type", lit("partner"))
    .withColumn("geometry_geojson", expr("ST_AsGeoJSON(geometry)"))
    .withColumn("h3_cell_id", expr("h3_longlatash3string(longitude, latitude, 8)"))
    .withColumn("partner_brand", when(F.lower(col("store_type")).contains("walmart"), "Walmart")
        .when(F.lower(col("store_type")).isin(["7-eleven", "speedway"]), "7-Eleven/Speedway")
        .when(F.lower(col("store_type")).contains("shaw"), "Shaw's").otherwise("Other"))
    .withColumn("partner_type", when(F.lower(col("store_type")).contains("walmart"), "Big Box")
        .when(F.lower(col("store_type")).isin(["7-eleven", "speedway"]), "Convenience")
        .when(F.lower(col("store_type")).contains("shaw"), "Grocery").otherwise("Other"))
    .withColumn("region", when((col("latitude") > 42.3) & (col("longitude") > -71.5), "Boston Metro")
        .when((col("latitude") > 42.0) & (col("longitude") > -72.0), "Greater Boston")
        .when(col("longitude") < -72.0, "Western MA").otherwise("Cape & Islands"))
)

viz_partners.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_partners")
print(f"✓ viz_partners: {viz_partners.count():,} rows")

## 5. Expansion Candidates

Joins candidates with pre-computed distances, partner proximity, and competitor counts.

In [ ]:
# Load base data
candidates = spark.table(f"{catalog}.{gold_schema}.candidates_finalized")
whitespace = spark.table(f"{catalog}.{silver_schema}.whitespace_locations").select(
    col("location_id").cast("string").alias("ws_id"),
    col("h3_cell_id").alias("ws_h3"),
    col("distance_to_nearest_current_store").alias("dist_miles"),
    col("nearest_store_id")
)
partner_iso = spark.table(f"{catalog}.{silver_schema}.isochrones_partners").select(
    col("location_id").alias("partner_id"), col("store_type").alias("partner_store_type"),
    col("city").alias("partner_city"), col("store_type").alias("partner_store_name"),
    col("drive_time_minutes").alias("partner_drive_time"), col("geometry").alias("partner_geometry")
)
competitors = spark.table(f"{catalog}.{gold_schema}.viz_competitors")

# Join whitespace for h3_cell_id and distance
df = candidates.join(whitespace, candidates["candidate_id"].cast("string") == whitespace["ws_id"], "left")
df = df.withColumn("h3_cell_id", F.coalesce(col("ws_h3"), expr("h3_longlatash3string(longitude, latitude, 8)")))
df = df.withColumn("min_distance_to_existing", F.coalesce(col("dist_miles"), lit(999.0)))
df = df.withColumn("nearest_existing_store", col("nearest_store_id")).drop("ws_id", "ws_h3", "dist_miles", "nearest_store_id")

# Partner proximity via spatial join
df_point = df.withColumn("pt", expr("ST_SetSRID(ST_Point(longitude, latitude), 4326)"))
df_partner = df_point.join(partner_iso, expr("ST_Contains(partner_geometry, pt)"), "left")
window_p = Window.partitionBy("candidate_id").orderBy("partner_drive_time")
df_partner = df_partner.withColumn("rn", F.row_number().over(window_p)).filter(col("rn") == 1).drop("rn", "pt", "partner_geometry")
df = df_partner.withColumn("within_partner_isochrone", col("partner_id").isNotNull())

# Competitor proximity
comp_point = competitors.withColumn("comp_pt", expr("ST_SetSRID(ST_Point(longitude, latitude), 4326)")).select(col("id").alias("comp_id"), "comp_pt")
df_comp = df.withColumn("cand_pt", expr("ST_SetSRID(ST_Point(longitude, latitude), 4326)")).crossJoin(comp_point)
df_comp = df_comp.withColumn("dist_m", expr("ST_DistanceSpheroid(cand_pt, comp_pt)")).filter(col("dist_m") <= 4828)  # 3 miles
comp_agg = df_comp.groupBy("h3_cell_id").agg(F.count("comp_id").alias("competitor_count"), F.round(F.min(col("dist_m") / 1609.34), 2).alias("nearest_competitor_miles"))
df = df.join(comp_agg, "h3_cell_id", "left").fillna({"competitor_count": 0, "nearest_competitor_miles": 99.0})

# Normalize sales score
stats = candidates.agg(F.min("predicted_annual_sales").alias("min_s"), F.max("predicted_annual_sales").alias("max_s")).collect()[0]
min_s, max_s = stats["min_s"], stats["max_s"]

viz_candidates = (
    df
    .withColumn("normalized_sales_score", (col("predicted_annual_sales") - lit(min_s)) / lit(max_s - min_s) if max_s > min_s else lit(0.5))
    .withColumn("fulfillment_strategy", when(col("within_partner_isochrone"), "partner").otherwise("new_store"))
    .withColumn("geometry", expr("ST_GeomFromWKT(h3_boundaryaswkt(h3_cell_id), 4326)"))
    .withColumn("geometry_geojson", expr("ST_AsGeoJSON(ST_GeomFromWKT(h3_boundaryaswkt(h3_cell_id), 4326))"))
    .withColumn("center_lat", expr("ST_Y(ST_GeomFromWKT(h3_centeraswkt(h3_cell_id), 4326))"))
    .withColumn("center_lon", expr("ST_X(ST_GeomFromWKT(h3_centeraswkt(h3_cell_id), 4326))"))
    .withColumn("sales_rank", F.row_number().over(Window.orderBy(F.desc("predicted_annual_sales"))))
    .withColumn("region", when((col("center_lat") > 42.3) & (col("center_lon") > -71.5), "Boston Metro").when((col("center_lat") > 42.0) & (col("center_lon") > -72.0), "Greater Boston").when(col("center_lon") < -72.0, "Western MA").otherwise("Cape & Islands"))
    .withColumn("cannibalization_risk", when(col("min_distance_to_existing") < 2.0, "High").when(col("min_distance_to_existing") < 3.0, "Medium").when(col("min_distance_to_existing") < 5.0, "Low").otherwise("None"))
    .withColumn("competitor_density", when(col("competitor_count") >= 5, "Saturated").when(col("competitor_count") >= 3, "Competitive").when(col("competitor_count") >= 1, "Moderate").otherwise("Low"))
    .withColumn("partner_brand", when(F.lower(col("partner_store_name")).contains("walmart"), "Walmart").when(F.lower(col("partner_store_name")).isin(["7-eleven", "speedway"]), "7-Eleven/Speedway").when(F.lower(col("partner_store_name")).contains("shaw"), "Shaw's").otherwise(None))
    .withColumn("partner_type", when(F.lower(col("partner_store_name")).contains("walmart"), "Big Box").when(F.lower(col("partner_store_name")).isin(["7-eleven", "speedway"]), "Convenience").when(F.lower(col("partner_store_name")).contains("shaw"), "Grocery").otherwise(None))
    .withColumn("city", when(col("partner_city").isNotNull(), col("partner_city")).otherwise(when(col("region") == "Boston Metro", "Boston Metro").otherwise("Massachusetts")))
    .fillna({"within_partner_isochrone": False})
)

viz_candidates.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_expansion_candidates")
print(f"✓ viz_expansion_candidates: {viz_candidates.count():,} rows")

## 6. Network Metrics

In [ ]:
existing = spark.table(f"{catalog}.{gold_schema}.current_stores_features_agg")
candidates = spark.table(f"{catalog}.{gold_schema}.candidates_finalized")

e_metrics = existing.agg(
    F.count("*").alias("total_existing_stores"),
    F.avg("population").alias("avg_store_population")
).collect()[0]

c_metrics = candidates.agg(
    F.count("*").alias("total_candidates"),
    F.avg("predicted_annual_sales").alias("avg_candidate_sales"),
    F.percentile_approx("predicted_annual_sales", 0.25).alias("sales_p25"),
    F.percentile_approx("predicted_annual_sales", 0.50).alias("sales_p50"),
    F.percentile_approx("predicted_annual_sales", 0.75).alias("sales_p75"),
    F.min("predicted_annual_sales").alias("sales_min"),
    F.max("predicted_annual_sales").alias("sales_max")
).collect()[0]

viz_network_metrics = spark.createDataFrame([{
    "total_existing_stores": int(e_metrics["total_existing_stores"]),
    "avg_store_population": float(e_metrics["avg_store_population"] or 0),
    "total_candidates": int(c_metrics["total_candidates"]),
    "avg_candidate_sales": float(c_metrics["avg_candidate_sales"] or 0),
    "sales_p25": float(c_metrics["sales_p25"] or 0),
    "sales_p50": float(c_metrics["sales_p50"] or 0),
    "sales_p75": float(c_metrics["sales_p75"] or 0),
    "sales_min": float(c_metrics["sales_min"] or 0),
    "sales_max": float(c_metrics["sales_max"] or 0),
    "last_updated": datetime.now()
}])

viz_network_metrics.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_network_metrics")
print(f"✓ viz_network_metrics: {e_metrics['total_existing_stores']} stores, {c_metrics['total_candidates']} candidates")

## 7. Optimization Results

Pre-compute for 8 parameter combinations (2×2×2 grid).

In [ ]:
def haversine_vec(lat1, lon1, lats2, lons2):
    R = 3959  # miles
    dlat, dlon = np.radians(lats2 - lat1), np.radians(lons2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lats2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def greedy_opt(cand_pdf, exist_lats, exist_lons, max_stores, min_new, min_exist):
    sorted_c = cand_pdf.sort_values('predicted_annual_sales', ascending=False).reset_index(drop=True)
    lats, lons, h3s = sorted_c['latitude'].values, sorted_c['longitude'].values, sorted_c['h3_cell_id'].values
    dist_exist = np.array([haversine_vec(lats[i], lons[i], exist_lats, exist_lons) for i in range(len(lats))])
    passes = np.all(dist_exist >= min_exist, axis=1)
    selected_h3, selected_idx = [], []
    for i in range(len(lats)):
        if len(selected_h3) >= max_stores or not passes[i]: continue
        if selected_idx and np.any(haversine_vec(lats[i], lons[i], lats[selected_idx], lons[selected_idx]) < min_new): continue
        selected_h3.append(h3s[i]); selected_idx.append(i)
    return selected_h3

# Load data
cand_pdf = spark.table(f"{catalog}.{gold_schema}.viz_expansion_candidates").select("h3_cell_id", "latitude", "longitude", "predicted_annual_sales").toPandas()
exist_pdf = spark.table(f"{catalog}.{gold_schema}.viz_existing_stores").select("latitude", "longitude").toPandas()
exist_lats, exist_lons = exist_pdf['latitude'].values, exist_pdf['longitude'].values

# Run optimization grid
results = []
for max_s, min_n, min_e in product([10, 50], [2.0, 3.0], [2.0, 3.0]):
    h3s = greedy_opt(cand_pdf, exist_lats, exist_lons, max_s, min_n, min_e)
    sales = cand_pdf[cand_pdf['h3_cell_id'].isin(h3s)]['predicted_annual_sales'].sum()
    results.append({"max_stores": max_s, "min_distance_new": min_n, "min_distance_existing": min_e,
                    "selected_h3_cells": h3s, "selected_count": len(h3s), "total_predicted_sales": float(sales), "computed_at": datetime.now()})

from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, ArrayType, StringType, TimestampType
schema = StructType([StructField("max_stores", IntegerType()), StructField("min_distance_new", DoubleType()), StructField("min_distance_existing", DoubleType()),
                     StructField("selected_h3_cells", ArrayType(StringType())), StructField("selected_count", IntegerType()),
                     StructField("total_predicted_sales", DoubleType()), StructField("computed_at", TimestampType())])
opt_df = spark.createDataFrame(results, schema)
opt_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_optimization_results")
print(f"✓ viz_optimization_results: {len(results)} combinations")

## 8. Column Comments for Genie

In [ ]:
# Add column comments to help with data understanding
comments = {
    f"{catalog}.{gold_schema}.viz_expansion_candidates": {
        "h3_cell_id": "Unique H3 hexagonal cell identifier",
        "predicted_annual_sales": "ML-predicted annual sales in USD",
        "sales_rank": "Rank by predicted sales (1 = highest)",
        "fulfillment_strategy": "Recommended approach: partner or new_store",
        "min_distance_to_existing": "Distance in miles to nearest existing store",
        "cannibalization_risk": "Risk level: High/Medium/Low/None",
        "competitor_count": "Pizza competitors within 3 miles",
        "competitor_density": "Competition level: Saturated/Competitive/Moderate/Low",
        "partner_brand": "Partner brand: Walmart/7-Eleven/Speedway/Shaw's",
        "region": "Geographic region in Massachusetts"
    },
    f"{catalog}.{gold_schema}.viz_existing_stores": {
        "store_number": "Unique store identifier",
        "annual_sales": "Actual annual sales in USD",
        "sales_rank": "Rank by sales (1 = highest)",
        "performance_tier": "Performance: Top Performer/Above Average/Average/Below Average",
        "region": "Geographic region in Massachusetts"
    },
    f"{catalog}.{gold_schema}.viz_partners": {
        "partner_brand": "Brand: Walmart/7-Eleven/Speedway/Shaw's",
        "partner_type": "Category: Big Box/Convenience/Grocery",
        "candidate_count_in_isochrone": "Expansion candidates in trade area",
        "total_candidate_sales_in_isochrone": "Total predicted sales in trade area"
    }
}

for table, cols in comments.items():
    for col_name, comment in cols.items():
        try: spark.sql(f"ALTER TABLE {table} ALTER COLUMN {col_name} COMMENT '{comment}'")
        except: pass

# Table comments
spark.sql(f"COMMENT ON TABLE {catalog}.{gold_schema}.viz_expansion_candidates IS 'Expansion opportunities ranked by ML-predicted sales'")
spark.sql(f"COMMENT ON TABLE {catalog}.{gold_schema}.viz_existing_stores IS 'Current Little Caesars store performance'")
spark.sql(f"COMMENT ON TABLE {catalog}.{gold_schema}.viz_partners IS 'Partner store co-location opportunities'")
print("✓ Column comments added")

## 9. Genie-Specific Tables

Creates simplified tables with only the columns needed for Genie natural language queries.

In [ ]:
# Create Genie-specific tables with only the columns needed for natural language queries
# These exclude geometry, h3_cell_id, and other technical columns

# Genie Existing Stores - simplified view of current LCE stores
genie_existing_stores = spark.table(f"{catalog}.{gold_schema}.viz_existing_stores").select(
    col("store_number"),
    col("store_type").alias("store_name"),  # Renamed for clarity
    "city", "state", "population", "annual_sales", "poi_count",
    "region", "sales_rank", "sales_percentile", "performance_tier"
)

genie_existing_stores.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.genie_existing_stores")
print(f"✓ genie_existing_stores: {genie_existing_stores.count():,} rows")

# Genie Expansion Candidates - simplified view with key decision columns
genie_expansion_candidates = spark.table(f"{catalog}.{gold_schema}.viz_expansion_candidates").select(
    "candidate_id", "city", "state", "population",
    "target_demographic_total", "total_poi_count", "human_activity_index", "urbanity",
    "predicted_annual_sales", "sales_rank",
    "partner_id", "partner_city", "within_partner_isochrone",
    "competitor_count", "normalized_sales_score",
    "fulfillment_strategy", "region", "cannibalization_risk", "competitor_density",
    "partner_brand", "partner_type"
)

genie_expansion_candidates.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.genie_expansion_candidates")
print(f"✓ genie_expansion_candidates: {genie_expansion_candidates.count():,} rows")

# Add table comments
spark.sql(f"COMMENT ON TABLE {catalog}.{gold_schema}.genie_existing_stores IS 'Simplified view of Little Caesars stores for Genie queries'")
spark.sql(f"COMMENT ON TABLE {catalog}.{gold_schema}.genie_expansion_candidates IS 'Simplified expansion opportunities for Genie queries'")

## 9. Genie Space (Optional)

Creates/updates a Databricks Genie space for natural language analytics.
Requires `sql_warehouse_id` parameter (Pro or Serverless SQL warehouse).

In [ ]:
# Create or Update Genie Space for Little Caesars Expansion Analytics
# Uses REST API: POST/PATCH /api/2.0/genie/spaces

import json, uuid

def gen_id():
    """Generate 32-char lowercase hex ID (required by Genie API)."""
    return uuid.uuid4().hex

if sql_warehouse_id:
    try:
        from databricks.sdk import WorkspaceClient
        w = WorkspaceClient()
        
        space_title = f"Little Caesars Expansion Analytics - {catalog}"
        space_description = "Natural language analytics for Little Caesars expansion planning in Massachusetts."
        
        # Tables - using genie-specific tables with clean schemas (no geometry/h3 columns)
        tables = sorted([
            {"identifier": f"{catalog}.{gold_schema}.genie_expansion_candidates", 
             "description": ["ML-predicted expansion opportunities ranked by sales potential"]},
            {"identifier": f"{catalog}.{gold_schema}.genie_existing_stores",
             "description": ["Current Little Caesars (LCE) store performance metrics"]}
        ], key=lambda x: x["identifier"])
        
        # 4 Sample questions (consistent with React app) - sorted by id
        sample_questions = sorted([
            {"id": gen_id(), "question": ["What are the top 5 expansion opportunities?"]},
            {"id": gen_id(), "question": ["Compare partner vs new store fulfillment strategy"]},
            {"id": gen_id(), "question": ["Show expansion candidates in Boston Metro with low competition"]},
            {"id": gen_id(), "question": ["Show existing store performance"]}
        ], key=lambda x: x["id"])
        
        # 4 Example question SQLs matching the sample questions (using genie_ tables) - sorted by id
        example_question_sqls = sorted([
            {"id": gen_id(), 
             "question": ["What are the top 5 expansion opportunities?"],
             "sql": [f"""SELECT candidate_id, city, predicted_annual_sales, fulfillment_strategy, partner_brand
FROM {catalog}.{gold_schema}.genie_expansion_candidates
ORDER BY predicted_annual_sales DESC
LIMIT 5"""]},
            {"id": gen_id(),
             "question": ["Compare partner vs new store fulfillment strategy"],
             "sql": [f"""SELECT fulfillment_strategy, COUNT(*) AS candidate_count, partner_brand, AVG(predicted_annual_sales) AS avg_predicted_annual_sales
FROM {catalog}.{gold_schema}.genie_expansion_candidates
WHERE fulfillment_strategy IS NOT NULL
GROUP BY fulfillment_strategy, partner_brand
ORDER BY avg_predicted_annual_sales DESC"""]},
            {"id": gen_id(),
             "question": ["Show expansion candidates in Boston Metro with low competition"],
             "sql": [f"""SELECT candidate_id, region, predicted_annual_sales, fulfillment_strategy, competitor_density
FROM {catalog}.{gold_schema}.genie_expansion_candidates
WHERE region ILIKE '%Boston Metro%' AND competitor_density ILIKE '%Low%'
ORDER BY predicted_annual_sales DESC"""]},
            {"id": gen_id(),
             "question": ["Show existing store performance"],
             "sql": [f"""SELECT store_number, city, annual_sales, sales_rank
FROM {catalog}.{gold_schema}.genie_existing_stores
WHERE store_number IS NOT NULL AND annual_sales IS NOT NULL
ORDER BY annual_sales DESC"""]}
        ], key=lambda x: x["id"])
        
        # Text instructions - sorted by id
        text_instructions = sorted([
            {"id": gen_id(), "content": [
                "LCE = Little Caesars. LCE stores refer to existing Little Caesars locations in genie_existing_stores. " +
                "When asked about best or top locations, ORDER BY predicted_annual_sales DESC. " +
                "fulfillment_strategy: partner means co-locate with existing partner, new_store means build new. " +
                "cannibalization_risk: High/Medium/Low/None based on distance to existing stores. " +
                "competitor_density: Saturated (5+), Competitive (3-4), Moderate (1-2), Low (0 within 3mi). " +
                "IMPORTANT: Always use LIMIT 5 for results. Select only 3-5 key columns for readability."
            ]}
        ], key=lambda x: x["id"])
        
        serialized_space = {
            "version": 2,
            "config": {
                "sample_questions": sample_questions
            },
            "data_sources": {
                "tables": tables
            },
            "instructions": {
                "text_instructions": text_instructions,
                "example_question_sqls": example_question_sqls
            }
        }
        
        # Check for existing space
        existing_id = None
        try:
            resp = w.api_client.do("GET", "/api/2.0/genie/spaces")
            for space in resp.get("spaces", []):
                if space.get("title") == space_title:
                    existing_id = space.get("space_id") or space.get("id")
                    break
        except Exception as e:
            print(f"Could not list spaces: {e}")
        
        if existing_id:
            # Update existing space with PATCH
            update_payload = {
                "title": space_title,
                "description": space_description,
                "serialized_space": json.dumps(serialized_space)
            }
            resp = w.api_client.do("PATCH", f"/api/2.0/genie/spaces/{existing_id}", body=update_payload)
            print(f"✓ Genie Space updated: {existing_id}")
        else:
            # Create new space with POST
            create_payload = {
                "warehouse_id": sql_warehouse_id,
                "title": space_title,
                "description": space_description,
                "serialized_space": json.dumps(serialized_space)
            }
            resp = w.api_client.do("POST", "/api/2.0/genie/spaces", body=create_payload)
            space_id = resp.get("space_id") or resp.get("id")
            print(f"✓ Genie Space created: {space_id}")
        
        print(f"  Access: SQL Editor → Genie → '{space_title}'")
            
    except Exception as e:
        print(f"⚠ Genie space error: {e}")
        import traceback
        traceback.print_exc()
else:
    print("Genie space skipped (no sql_warehouse_id)")

## Summary

In [ ]:
print("=" * 50)
print("VISUALIZATION LAYER COMPLETE")
print("=" * 50)
for t in ["viz_h3_grid", "viz_competitors", "viz_existing_stores", "viz_partners", "viz_expansion_candidates", "viz_network_metrics", "viz_optimization_results", "genie_existing_stores", "genie_expansion_candidates"]:
    try: print(f"  ✓ {t}: {spark.table(f'{catalog}.{gold_schema}.{t}').count():,} rows")
    except: print(f"  ✗ {t}: not found")